In [ ]:
# Mount ADLS Gen2
tiers = ['bronze', 'silver', 'gold']
adls_paths = {tier: f"abfss://{tier}@datalakehan.dfs.core.windows.net/" for tier in tiers}

# paths
bronze_adls = adls_paths['bronze']
silver_adls = adls_paths['silver']
gold_adls = adls_paths['gold']

dbutils.fs.ls(bronze_adls)
dbutils.fs.ls(silver_adls)
dbutils.fs.ls(gold_adls)

[FileInfo(path='abfss://gold@datalakehan.dfs.core.windows.net/earthquake_events_gold/', name='earthquake_events_gold/', size=0, modificationTime=1773414884000)]

In [ ]:
bronze_adls

'abfss://bronze@datalakehan.dfs.core.windows.net/'

In [ ]:
import requests
import json
from datetime import date, timedelta

In [ ]:
# get the base parameters
dbutils.widgets.text("start_date", "")
start_date = dbutils.widgets.get("start_date")
dbutils.widgets.text("end_date", "")
end_date = dbutils.widgets.get("end_date")

In [ ]:
# construct the api url with startand end dates provided by Data Factory
url = f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime={start_date}&endtime={end_date}"

try:
    # making the GET request
    response = requests.get(url)

    # status of the response
    response.raise_for_status() 
    data = response.json().get('features', [])

    if not data:
        print('No data found for the given date range.')
    else:
        # specify the ADLS path
        file_path = f'{bronze_adls}/{start_date}_earthquake_data.json'

        # save the json data
        json.dumps(data, indent=4)
        dbutils.fs.put(file_path, json.dumps(data), overwrite=True)  # db utils, fs = file system
        print(f'Data saved to {file_path}')
except requests.exceptions.RequestException as e:
    print(f"Error getting data from API: {e}")

Wrote 8075831 bytes.
Data saved to abfss://bronze@datalakehan.dfs.core.windows.net//_earthquake_data.json


In [ ]:
data[0]

{'type': 'Feature',
 'properties': {'mag': 1.13,
  'place': '6 km NW of The Geysers, CA',
  'time': 1773492634280,
  'updated': 1773492732161,
  'tz': None,
  'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/nc75327267',
  'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=nc75327267&format=geojson',
  'felt': None,
  'cdi': None,
  'mmi': None,
  'alert': None,
  'status': 'automatic',
  'tsunami': 0,
  'sig': 20,
  'net': 'nc',
  'code': '75327267',
  'ids': ',nc75327267,',
  'sources': ',nc,',
  'types': ',nearby-cities,origin,phase-data,',
  'nst': 24,
  'dmin': 0.009679,
  'rms': 0.01,
  'gap': 43,
  'magType': 'md',
  'type': 'earthquake',
  'title': 'M 1.1 - 6 km NW of The Geysers, CA'},
 'geometry': {'type': 'Point',
  'coordinates': [-122.798500061035, 38.8173332214355, 2.51999998092651]},
 'id': 'nc75327267'}

In [ ]:
# defining variables
output_data = {
    "start_date": start_date,
    "end_date": end_date,
    "bronze_adls": bronze_adls,
    "silver_adls": silver_adls,
    "gold_adls": gold_adls
}

# serialize the dictionary to JSON
output_json = json.dumps(output_data)

# log the serialised JSON for debugging
print(f"Serialised JSON: {output_json}")

# Return the JSON as a string
dbutils.notebook.exit(output_json)